# 4.03 从零写 GPT：语言模型、采样与 KV Cache

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/04_transformer/4.03_gpt_from_scratch.ipynb)


## 为什么要学

VLA 的"L"（语言）部分是一个预训练好的语言模型（π0 用 Gemma-2B，SmolVLA 用 SmolLM2，FluxVLA 支持 Qwen3-VL 等）。它们全都是 **decoder-only 的 GPT 结构**：输入 token 序列，预测下一个 token。

本课用上一课的 Block 搭一个小 GPT，让它**读本项目的全部课程文本**，学会"写课程"。然后实现 **KV cache**：这是 π0 推理时"前缀只算一次、去噪 10 步复用"的关键技术。参考：Karpathy 的 [nanoGPT](https://github.com/karpathy/nanoGPT)。

> 上一课写的组件已整理在 [`vla_ladder/models/transformer.py`](../../vla_ladder/models/transformer.py)，本课直接导入使用。

---

## 1. 语言模型的目标

把文本切成 token 序列 $x_1,\dots,x_T$，语言模型学习条件分布：

$$
p(x_1,\dots,x_T)=\prod_{t=1}^T p(x_t\mid x_{<t})
$$

训练：每个位置都预测**下一个** token，用交叉熵（1.07 课）。因果掩码保证位置 $t$ 看不到 $t$ 之后的内容，所以一次前向就能并行训练所有位置。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import vla_ladder
from vla_ladder.models.transformer import Block, RMSNorm, rope_cos_sin
from vla_ladder.utils import set_seed, get_device, count_params

device = get_device()
lesson_dir = Path(vla_ladder.__file__).resolve().parents[1] / "lessons"
text = "\n".join(p.read_text(encoding="utf-8") for p in sorted(lesson_dir.rglob("*.md")))
print(f"语料：{len(list(lesson_dir.rglob('*.md')))} 篇课程，共 {len(text):,} 个字符")
print(text[:200])

## 2. 分词：字符级

真实模型用 BPE/SentencePiece 子词分词（PaliGemma 词表 257,152）。这里用最简单的**字符级**：每个不同的字符是一个 token。

### 练习 1：字符级分词器

In [ ]:
class CharTokenizer:
    def __init__(self, text):
        # TODO: chars = 排序后的不重复字符；self.stoi: 字符->id；self.itos: id->字符
        raise NotImplementedError

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class CharTokenizer:
    def __init__(self, text):
        chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for c, i in self.stoi.items()}

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)

In [ ]:
# ✅ 自测
tok = CharTokenizer(text)
V = len(tok.stoi)
assert tok.decode(tok.encode("注意力机制")) == "注意力机制"
data = torch.tensor(tok.encode(text), dtype=torch.long)
n_train = int(0.95 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"通过 ✅ 词表大小 {V}，训练 {len(train_data):,} 个 token，验证 {len(val_data):,} 个")

### 练习 2：随机取训练窗口

从数据中随机取 `B` 个长度为 `L+1` 的片段，输入 `x` 是前 `L` 个，目标 `y` 是后 `L` 个（错开一位）。

In [ ]:
def get_batch(data, B, L):
    # TODO: ix = torch.randint(len(data) - L - 1, (B,))；x = stack(data[i:i+L])；y = stack(data[i+1:i+L+1])
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def get_batch(data, B, L):
    ix = torch.randint(len(data) - L - 1, (B,))
    x = torch.stack([data[i : i + L] for i in ix])
    y = torch.stack([data[i + 1 : i + L + 1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
# ✅ 自测
xb, yb = get_batch(train_data, 4, 16)
assert xb.shape == yb.shape == (4, 16) and torch.equal(xb[:, 1:], yb[:, :-1])
print("通过 ✅")

---

## 3. 搭建 GPT

结构：`token embedding (× √d) → N 个 Block（因果掩码 + RoPE） → 最终 RMSNorm → 与 embedding 共享权重的输出层`。

### 练习 3：GPT 的前向传播

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, dim=192, depth=4, n_heads=4, head_dim=48, mlp_hidden=768):
        super().__init__()
        self.dim, self.head_dim = dim, head_dim
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, n_heads, head_dim, mlp_hidden) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        nn.init.normal_(self.embed.weight, std=0.02)   # 很重要！见下方"自测"后的说明

    def forward(self, idx):
        """idx: (B, L) -> logits (B, L, vocab)"""
        B, L = idx.shape
        # TODO:
        # 1) x = embed(idx) * sqrt(dim)
        # 2) RoPE：pos = arange(L) 扩展成 (B, L)，cos_sin = rope_cos_sin(pos, head_dim)
        # 3) 因果掩码 mask = tril(ones(L, L)).bool()
        # 4) 依次过每个 block：x, _ = blk(x, mask, cos_sin)
        # 5) logits = norm(x) @ embed.weight.T      （输出层与输入 embedding 共享权重）
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class GPT(nn.Module):
    def __init__(self, vocab_size, dim=192, depth=4, n_heads=4, head_dim=48, mlp_hidden=768):
        super().__init__()
        self.dim, self.head_dim = dim, head_dim
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, n_heads, head_dim, mlp_hidden) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, idx):
        """idx: (B, L) -> logits (B, L, vocab)"""
        B, L = idx.shape
        x = self.embed(idx) * math.sqrt(self.dim)
        pos = torch.arange(L, device=idx.device)[None].expand(B, -1)
        cos_sin = rope_cos_sin(pos, self.head_dim)
        mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=idx.device))
        for blk in self.blocks:
            x, _ = blk(x, mask, cos_sin)
        return self.norm(x) @ self.embed.weight.T

In [ ]:
# ✅ 自测：初始 loss 应约为 log(V)（随机猜）
set_seed(0)
model = GPT(V).to(device)
logits = model(xb)
loss0 = F.cross_entropy(logits.reshape(-1, V), yb.reshape(-1)).item()
print(f"参数量 {count_params(model) / 1e6:.2f} M；初始 loss {loss0:.2f}，log(V) = {math.log(V):.2f}")
assert logits.shape == (4, 16, V) and abs(loss0 - math.log(V)) < 2.5
print("通过 ✅")

> **为什么要 `normal_(std=0.02)`？** `nn.Embedding` 默认用标准正态 $\mathcal N(0,1)$ 初始化。由于输出层与 embedding 共享权重，logits = (单位尺度的特征) · (embedding)ᵀ，每个 logit 的标准差约为 $\sqrt{d}\approx 14$，softmax 极度"自信"，初始 loss 会达到 100 以上（你可以注释掉那一行试试）。把 embedding 初始化得小一些，初始 loss 就回到 $\log V$ 附近。这是 1.10 课"检查初始 loss"技巧的一次真实应用。

## 4. 训练（并观察过拟合）

我们的语料只有二十多万个字符，而模型有两百多万参数，**很容易过拟合**：训练集 loss 一直降，验证集 loss 先降后升。下面每 100 步评估一次验证集，并保存**验证 loss 最低**时的参数（早停，early stopping）。

In [ ]:
import copy

@torch.no_grad()
def estimate_loss(model, data, n_batches=10, B=64, L=128):
    model.eval()
    losses = [F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item() for x, y in (get_batch(data, B, L) for _ in range(n_batches))]
    model.train()
    return float(np.mean(losses))

def train_gpt(model, steps=1500, B=64, L=128, lr=1e-3, eval_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    log = {"step": [], "train": [], "val": []}
    best_val, best_state, t0 = float("inf"), None, time.time()
    for step in range(steps):
        x, y = get_batch(train_data, B, L)
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if step % eval_every == 0 or step == steps - 1:
            tr, va = estimate_loss(model, train_data), estimate_loss(model, val_data)
            log["step"].append(step); log["train"].append(tr); log["val"].append(va)
            if va < best_val:
                best_val, best_state = va, copy.deepcopy(model.state_dict())
            if step % 500 == 0 or step == steps - 1:
                print(f"step {step:5d}  train {tr:.3f}  val {va:.3f}  ({time.time() - t0:.0f}s)")
    return log, best_state, best_val

log, best_state, best_val = train_gpt(model)
plt.figure(figsize=(5, 3)); plt.plot(log["step"], log["train"], label="train"); plt.plot(log["step"], log["val"], label="val")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("GPT: train vs val (overfitting)"); plt.show()
model.load_state_dict(best_state)
print(f"已加载验证集最优的参数（val loss {best_val:.3f}）")

> **这对 VLA 微调同样重要**：你在 ALOHA 上采的几十到几百条示教，相对于 30 亿参数的 π0 是非常小的数据集。训练步数太多，模型会"背下"示教轨迹，部署时对微小的扰动都很脆弱。openpi 的微调通常只跑几千到三万步，并且要在真机上比较不同 checkpoint 的成功率，**loss 最低不等于真机效果最好**。

## 5. 采样生成

每次取最后一个位置的 logits，除以**温度** $T$（$T<1$ 更保守、$T>1$ 更随机），可选只保留概率最高的 $k$ 个（top-k），然后采样。

In [ ]:
@torch.no_grad()
def generate(model, prompt, n_new=200, temperature=0.8, top_k=20):
    idx = torch.tensor([tok.encode(prompt)], device=device)
    for _ in range(n_new):
        logits = model(idx[:, -128:])[:, -1] / temperature      # 只用最后 128 个 token 作为上下文（与训练长度一致）
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
        nxt = torch.multinomial(torch.softmax(logits, -1), 1)
        idx = torch.cat([idx, nxt], 1)
    return tok.decode(idx[0].tolist())

model.eval()
print(generate(model, "## 1. 注意力"))

一个只有两百多万参数、训练一两分钟的模型，已经学会了 Markdown 的格式、代码和公式的写法以及一些"像样"的词组（当然内容是胡言乱语）。把参数放大上千倍、数据放大上百万倍，就是 Gemma。

---

## 6. KV Cache：推理加速的关键

上面的 `generate` 每生成一个 token，都要把**整个序列**重新算一遍，复杂度 $O(T^2)$。但因果掩码下，**旧 token 的 K、V 永远不会变**（它们看不到新 token）。所以可以把每层的 K、V 缓存起来，每次只为**新 token** 计算 q/k/v，再与缓存拼接：

```text
没有 cache：第 t 步计算 t 个 token 的 q,k,v   → 总计 O(T²)
有 cache  ：第 t 步只计算 1 个新 token 的 q,k,v，拼上缓存的 K,V → 总计 O(T)
```

**π0 的推理就是这个思路的变体**：前缀（图像+文字）先算一遍并缓存 KV；之后 10 步去噪，每步只计算 50 个动作 token，它们通过注意力读取缓存的前缀 KV。前缀看不到动作（4.01 课的掩码），所以缓存在 10 步之间完全不变。

### 练习 4：带缓存的单头注意力

In [ ]:
def attention_with_cache(q_new, k_new, v_new, cache=None):
    """
    q_new, k_new, v_new: (B, l, d) 新 token 的 q/k/v；cache: None 或 (K_past (B, P, d), V_past (B, P, d))
    新 token i 可以看：全部历史 token + 新 token 中位置 <= i 的
    返回 (输出 (B, l, d), 新缓存 (K_all, V_all))
    """
    # TODO:
    # 1) K = cat([K_past, k_new])，V 同理（cache 为 None 时就是 k_new、v_new）
    # 2) P = 历史长度；mask[i, j] = j <= P + i   （形状 (l, P+l)）
    # 3) out = softmax(q K^T / sqrt(d) 加掩码) V
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def attention_with_cache(q_new, k_new, v_new, cache=None):
    """
    q_new, k_new, v_new: (B, l, d) 新 token 的 q/k/v；cache: None 或 (K_past (B, P, d), V_past (B, P, d))
    新 token i 可以看：全部历史 token + 新 token 中位置 <= i 的
    返回 (输出 (B, l, d), 新缓存 (K_all, V_all))
    """
    if cache is not None:
        K, V = torch.cat([cache[0], k_new], 1), torch.cat([cache[1], v_new], 1)
    else:
        K, V = k_new, v_new
    l, total = q_new.shape[1], K.shape[1]
    P = total - l
    mask = torch.arange(total, device=q_new.device)[None, :] <= (P + torch.arange(l, device=q_new.device))[:, None]
    scores = q_new @ K.transpose(1, 2) / math.sqrt(q_new.shape[-1])
    scores = scores.masked_fill(~mask, float("-inf"))
    return torch.softmax(scores, -1) @ V, (K, V)

In [ ]:
# ✅ 自测：逐个 token 用缓存计算 == 一次性用因果掩码计算
torch.manual_seed(0)
q, k, v = torch.randn(3, 2, 9, 16).unbind(0)
full, _ = attention_with_cache(q, k, v)                     # 无缓存：一次算完 9 个
outs, cache = [], None
out, cache = attention_with_cache(q[:, :4], k[:, :4], v[:, :4], cache)   # 先算一个 4 token 的"前缀"
outs.append(out)
for t in range(4, 9):                                         # 再逐个生成
    out, cache = attention_with_cache(q[:, t : t + 1], k[:, t : t + 1], v[:, t : t + 1], cache)
    outs.append(out)
assert torch.allclose(torch.cat(outs, 1), full, atol=1e-5)
print("通过 ✅")

配套代码包里的 `vla_ladder.models.transformer.GPT` 已经在每一层实现了 KV cache（`MultiHeadAttention.forward` 的 `kv_cache` 参数）。把训练好的权重拷贝过去，比较有无 cache 的生成速度：

In [ ]:
from vla_ladder.models.transformer import GPT as CachedGPT
fast = CachedGPT(V, dim=192, depth=4, n_heads=4, head_dim=48, mlp_hidden=768).eval()
fast.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})   # 两个模型的参数名完全一致
fast = fast.cpu()                                            # 在 CPU 上计时，差别更明显（GPU 上小模型的耗时主要是启动开销）
prompt = torch.tensor([tok.encode("## 2. ")])
for use_cache in [False, True]:
    torch.manual_seed(0)
    t0 = time.time(); out = fast.generate(prompt, 300, temperature=0.8, top_k=20, use_cache=use_cache); dt = time.time() - t0
    print(f"use_cache={use_cache}: {dt:.2f}s")
print(tok.decode(out[0].tolist())[:300])

---

## 小结

- GPT = token embedding + 因果 Transformer + 输出层（与 embedding 共享权重），训练目标是预测下一个 token。
- 采样：温度 + top-k。
- KV cache：缓存旧 token 的 K、V，生成复杂度从 $O(T^2)$ 降到 $O(T)$。π0 用它缓存图像+文字前缀，10 步去噪只算动作 token。

**思考题**：PaliGemma 的词表有 25.7 万个 token，embedding 矩阵 257152×2048 就有 5.3 亿参数。π0 输出动作根本不需要语言模型的输出层（`lm_head`），openpi 里是怎么处理的？（提示：搜索 `gemma_expert.model.embed_tokens = None`，以及 π0 的前向里用了 `language_model` 的哪些部分。）